In [3]:
import os, sys, glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from scipy.stats import ttest_ind

REPO = '/grid/koo/home/pmantill/projects/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra'
sys.path.insert(0, os.path.join(REPO, 'eigen-interactions'))
import eigen_steering
from eigen_steering import EigenMap, AlphaGenome, MPRAHead, AlphaGenomeMPRA, remove_all_heads

# Local layout: weights at pytorch_base_model/, ckpts at models/{name}/best_stage2.pt
eigen_steering.WEIGHTS_PATH = os.path.join(REPO, 'pytorch_base_model', 'model_fold_0.safetensors')
CKPT_DIR = os.path.join(REPO, 'models')

CT = {'K562': 'K562_v6_do075', 'HepG2': 'HepG2_v6_do03'}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- libraries: sequences + precomputed attributions ---
df = pd.read_csv(os.path.join(REPO, 'data', 'joint_library_combined.csv'))
df = df.dropna(subset=['sequence'] + [f'{ct}_log2FC' for ct in CT]).reset_index(drop=True)

em = EigenMap(model_names=CT, device=DEVICE)
em.load_from_dataframe(df, seq_col='sequence')
em.set_actual({ct: df[f'{ct}_log2FC'].values for ct in CT})

raw = np.load(os.path.join(REPO, 'genomic_targets', 'data', 'deeplift_attributions.npz'))
df_full = pd.read_csv(os.path.join(REPO, 'data', 'joint_library_combined.csv'))
seq_valid = df_full['sequence'].notna(); n = seq_valid.sum()
keep = df_full.loc[seq_valid, ['sequence'] + [f'{ct}_log2FC' for ct in CT]].notna().all(axis=1).values
del df_full

ohe = em.X.numpy()
for ct in CT:
    hyp = raw[f'attr_{ct}'][:n][keep]
    em.attr_hyp[ct] = hyp; em.attr[ct] = hyp * ohe
    em.importance[ct] = em.attr[ct].sum(axis=1)
    em.predictions[ct] = raw[f'predictions_{ct}'][:n][keep]

# --- eigen cache + focused library ---
with open(os.path.join(REPO, 'genomic_targets', 'data', 'eigen_analysis.pkl'), 'rb') as f:
    cached = pickle.load(f)
ei1_var = cached['ei1_var']; corrs = cached['corrs']; eixr = ei1_var * corrs

with open(os.path.join(REPO, 'virtual_perturbations/libraries/hippo_target_library.pkl'), 'rb') as f:
    target_lib = pickle.load(f)
focus_idx = target_lib['df']['seq_idx'].values

# --- per-replicate MPRA stats ---
MPRA_DIR = os.path.join(REPO, 'data', 'full_joint_mpra')
COLS = ['cell', 'rep', 'name', 'dna', 'rna', 'ratio', 'log2_ratio', 'n_bc']
def _load(cell):
    return pd.concat([pd.read_csv(f, sep='\t', header=None, names=COLS)
                      for f in sorted(glob.glob(f'{MPRA_DIR}/{cell}/{cell}_rep*.tsv'))],
                     ignore_index=True)
hep_w = _load('HepG2').pivot_table(index='name', columns='rep', values='log2_ratio')
k_w   = _load('K562' ).pivot_table(index='name', columns='rep', values='log2_ratio')
shared = hep_w.index.intersection(k_w.index)
H, K = hep_w.loc[shared].values, k_w.loc[shared].values
ok = np.isfinite(H).all(1) & np.isfinite(K).all(1)
_, pv = ttest_ind(H[ok], K[ok], axis=1, equal_var=False)
diffs = df['name'].map(dict(zip(shared[ok], H[ok].mean(1) - K[ok].mean(1)))).values.astype(float)
pvals = df['name'].map(dict(zip(shared[ok], pv))).values.astype(float)

# --- heavy load: models on GPU (ckpts live at models/{name}/best_stage2.pt) ---
def _load_model_patched(self, ct, squeeze=False):
    name = self.model_names[ct]
    ckpt_path = os.path.join(CKPT_DIR, name, 'best_stage2.pt')
    print(f"  Loading {ct}: {ckpt_path}")
    enc = AlphaGenome.from_pretrained(eigen_steering.WEIGHTS_PATH, device='cpu')
    remove_all_heads(enc)
    hd = MPRAHead()
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    enc.load_state_dict(ckpt['model_state_dict'], strict=False)
    hd.load_state_dict(ckpt['head_state_dict'])
    return AlphaGenomeMPRA(enc, hd, squeeze=squeeze).to(self.device).eval()

EigenMap._load_model = _load_model_patched
models = em._load_models()

print(f'\n{len(df)} seqs on {DEVICE} | EIxr n={len(eixr)} | focus={len(focus_idx)} | mpra={np.isfinite(diffs).sum()}')
for ct, m in models.items():
    print(f'  {ct}: {sum(p.numel() for p in m.parameters())/1e6:.2f}M params on {next(m.parameters()).device}')

EigenMap: ['K562', 'HepG2'], models={'K562': 'K562_v6_do075', 'HepG2': 'HepG2_v6_do03'}
Loaded 56975 sequences, X shape: torch.Size([56975, 4, 281])
  Loading K562: /grid/koo/home/pmantill/projects/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra/models/K562_v6_do075/best_stage2.pt
  Loading HepG2: /grid/koo/home/pmantill/projects/Virtual_Experiments/Hippo_axis/Hippo_dependency_mpra/models/HepG2_v6_do03/best_stage2.pt

56975 seqs on cuda | EIxr n=56975 | focus=1059 | mpra=56962
  K562: 408.64M params on cuda:0
  HepG2: 408.64M params on cuda:0
